In [70]:
import torch
from torch import nn
import cv2

from diffusers.models import AutoencoderKL

import xformers.ops as xops
from rotary_embedding_torch import RotaryEmbedding

In [ ]:
device = "cuda"
batch_size = 1
latent_dim = 4
embed_dim = 64
hidden_dim = 256
num_heads = 8

In [3]:
z = torch.randn((batch_size, 4, 64, 64), device=device)

In [4]:
ffn1 = nn.Sequential(
    nn.Linear(4, embed_dim),
).to(device)

ffn2 = nn.Sequential(
    nn.Linear(embed_dim, hidden_dim),
    nn.GELU(),
    nn.Linear(hidden_dim, embed_dim),
).to(device)

rope = RotaryEmbedding(dim=32).to(device)

In [ ]:
tokens = z.flatten(start_dim=2).permute(0, 2, 1)
h1 = ffn1(tokens)[:, None, :, :].repeat(
    1, num_heads, 1, 1
)  # (batch, heads, seq len, dimension of head)
h2_qk = rope.rotate_queries_or_keys(h1).permute(
    (0, 2, 1, 3)
)  # (batch, seq len, heads, dimension of head)
h2_v = h1.permute((0, 2, 1, 3))  # (batch, seq len, heads, dimension of head)
h3 = xops.memory_efficient_attention(h2_qk, h2_qk, h2_v)
h4 = ffn2(h3)

In [71]:
class ConditionalEncoding(nn.Module):
    def __init__(self, encoding_dim, hidden_dim=512):
        assert (encoding_dim - 1) % 2 == 0, "Encoding Dim Invalid Size"
        super().__init__()
        self.length = (encoding_dim - 1) // 2
        self.ffn = nn.Sequential(
            nn.Linear(encoding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, encoding_dim),
        )

    def positional_encoding(self, x):
        device = x.device
        B, _ = x.shape
        pe = torch.zeros((B, 2 * self.length + 1), device=device)
        pe[:, 0] = x.T
        xh = 2 ** torch.arange(self.length, device=device) * torch.pi * x
        pe[:, 1::2] = torch.sin(xh)
        pe[:, 2::2] = torch.cos(xh)
        return pe
    
    def forward(self, x):
        with torch.no_grad():
            pe = self.positional_encoding(x)
        return self.ffn(pe)


class AdaLayerNorm(nn.Module):
    def __init__(self, normalized_shape, cond_dim, eps=1e-5):
        super().__init__()
        self.norm = nn.LayerNorm(normalized_shape, elementwise_affine=False, eps=eps)
        self.modulation = nn.Linear(cond_dim, 2 * normalized_shape)

    def forward(self, x, cond):
        x = self.norm(x)
        gamma_beta = self.modulation(cond)
        gamma, beta = gamma_beta.chunk(2, dim=-1)
        gamma = gamma.unsqueeze(1)
        beta = beta.unsqueeze(1)
        return gamma * x + beta

In [72]:
pe1 = ConditionalEncoding(21, hidden_dim=128)
ln1 = AdaLayerNorm(512, 63)

In [ ]:
a = torch.randn((10, 3), requires_grad=True)
ax = pe1(a[:, [0]])
ay = pe1(a[:, [1]])
az = pe1(a[:, [2]])
a_enc = torch.hstack([ax, ay,  az])

In [77]:
d = torch.randn((10, 4096, 512), requires_grad=True)
ln1(d, a_enc).shape

torch.Size([10, 4096, 512])